# DroneFuse - T2: import DroneRFc-MM into Kaggle (CPU, Internet On)

DroneRFc-MM (Yu et al., Science Data Bank, doi:10.57760/sciencedb.j00173.00094, CC BY-SA 4.0) records six DJI
drones with a PTZ camera, a fisheye camera, an RF receiver (IQ), a LiDAR, a mmWave radar and a four-channel
microphone array, on one afternoon at one site. Sessions A1, E1 and G1 were flown again ~1.5 h later
(A1-2, E1-2, G1-2), which gives a cross-recording test like the one on MMAUD.

The release is 75.6 GB, more than one notebook can save (~20 GB), so it is imported in **parts** of at most
15 GB. Each run downloads one part **directly from Science Data Bank into this session** (nothing passes
through your computer), with 16 parallel connections, checks every file against the MD5 of the release and
keeps the files unchanged under their release paths.

| Part | Content | Size |
|---|---|---|
| `core` | ground truth, labels, audio, radar, PTZ and fisheye video, sample clips, code | ~13 GB |
| `lidar_1`, `lidar_2` | LiDAR point clouds | 12.4 + 6.4 GB |
| `rf_1` ... `rf_4` | RF IQ archives (.rar) | 9-12.5 GB each |

**Kaggle settings:** Accelerator = **None** (CPU). Internet = **On**.
**Inputs to add:** `dronefuse-code` (latest). If an earlier run of the *same part* stopped early, add its output
too: the files it already holds are skipped.
**Run time:** about 1 h per part at the 3.5 MB/s measured from China; the first cell after the settings
measures the speed from Kaggle.

In [ ]:
import os
import sys
from pathlib import Path

# The DroneFuse code is attached as a Kaggle dataset. Find the `dronefuse`
# package wherever Kaggle placed it, and make it importable.
def find_file(name, root='/kaggle/input'):
    # os.walk with followlinks=True also enters inputs that Kaggle mounts as links.
    for folder, _, files in os.walk(root, followlinks=True):
        if name in files:
            return Path(folder) / name
    return None


package = find_file('rosbag_index.py')
if package is None:
    attached = sorted(str(p) for p in Path('/kaggle/input').glob('*/*'))[:20]
    raise RuntimeError('The dronefuse-code dataset is not attached to this notebook. In the editor: '
                       'Add Input -> Your Datasets -> dronefuse-code, then Save & Run All again. '
                       f'Attached now: {attached or "nothing"}')
if (package.parent / '__init__.py').exists() and package.parent.name == 'dronefuse':
    code_root = package.parent.parent
else:
    # Kaggle sometimes flattens an uploaded folder, so the files sit directly in
    # the dataset with no `dronefuse/` folder around them. In that case, link the
    # folder that holds rosbag_index.py under the name `dronefuse`.
    flat = package
    code_root = Path('/kaggle/working/code')
    code_root.mkdir(parents=True, exist_ok=True)
    if not (code_root / 'dronefuse').exists():
        os.symlink(flat.parent, code_root / 'dronefuse')

sys.path.insert(0, str(code_root))
import dronefuse
print('DroneFuse code loaded from', Path(dronefuse.__file__).parent)

In [ ]:
# ---- Settings (the only cell you should need to edit) ----
INPUT = Path('/kaggle/input')
PART = 'core'                      # 'core', 'lidar_1', 'lidar_2', 'rf_1', 'rf_2', 'rf_3' or 'rf_4'
OUTPUT = Path('/kaggle/working/rfcmm')
WORKERS = 16                       # parallel connections
TIME_BUDGET_MINUTES = 11 * 60

In [ ]:
import json
from dronefuse import environment, rfcmm
from dronefuse.runner import StepRunner

OUTPUT.mkdir(parents=True, exist_ok=True)
runner = StepRunner(OUTPUT, TIME_BUDGET_MINUTES)
summary = {'part': PART, 'environment': runner.run('measure the machine', environment.collect)}
files = runner.run('read the Science Data Bank file list', rfcmm.list_files)
parts = rfcmm.plan_parts(files or [])
for name, fs in parts.items():
    print(f"  {name:8s} {len(fs):3d} files {sum(f['size'] for f in fs) / 1e9:6.2f} GB")
assert PART in parts, f"PART must be one of {list(parts)}"
# an earlier run keeps its files next to its import_summary.json
earlier_roots = sorted({p.parent for p in INPUT.rglob('import_summary.json')
                        if json.loads(p.read_text()).get('part') == PART})
print('earlier runs of this part attached:', [str(p) for p in earlier_roots] or 'none')
summary['speed_test'] = runner.run('speed test from this session', rfcmm.speed_test, files, 60, WORKERS)
print(summary['speed_test'])
if summary['speed_test']:
    gb = sum(f['size'] for f in parts[PART]) / 1e9
    print(f"expected time for part {PART} ({gb:.1f} GB): {gb * 1000 / summary['speed_test']['mb_per_s'] / 60:.0f} min")

In [ ]:
# The import: one file at a time, each with WORKERS parallel range requests, MD5-checked.
budget = runner.minutes_left() * 60
result = runner.run(f'import part {PART}', rfcmm.import_part, PART, OUTPUT, files, earlier_roots, budget, WORKERS)
summary['import'] = result
with open(OUTPUT / 't2_summary.json', 'w') as f:
    json.dump(summary, f, indent=1, default=str)

In [ ]:
# Report
r = result or {}
held = [p for p in OUTPUT.rglob('*') if p.is_file() and p.name not in ('t2_summary.json', 't2_report.txt')]
lines = [f"PART {PART}: {r.get('files', '?')} files, {r.get('gb', '?')} GB in the release",
         f"IN THIS OUTPUT: {len(held)} files, {sum(p.stat().st_size for p in held) / 1e9:.2f} GB",
         f"SPEED TEST: {json.dumps(summary.get('speed_test'))}",
         f"DONE NOW: {len(r.get('done', {}))}   ALREADY IN AN EARLIER RUN: {len(r.get('skipped_existing', []))}",
         'NOT REACHED (time budget): ' + (', '.join(r.get('not_reached', [])) or 'none'),
         'FAILED: ' + (json.dumps(r.get('failed')) if r.get('failed') else 'none')]
speeds = sorted(v['mb_per_s'] for v in r.get('done', {}).values() if v.get('bytes', 0) > 50e6)
if speeds:
    lines.append(f"DOWNLOAD SPEED (files > 50 MB): median {speeds[len(speeds) // 2]} MB/s")
failed = runner.failed()
lines.append('STEPS: ' + ('all succeeded' if not failed else 'not succeeded: ' + ', '.join(failed)))
report = '\n'.join(lines)
(OUTPUT / 't2_report.txt').write_text(report)
print(report)

## After the run

1. Send back `rfcmm/t2_report.txt`.
2. Open this notebook's saved version -> **Output** -> **New Dataset**, name it **`dronefuse-rfcmm-<part>`**
   (e.g. `dronefuse-rfcmm-core`, private).
3. If the report lists files as *not reached* or *failed*, attach that dataset as an input and run the same part
   again; only the missing files are fetched.
4. For the next part, change `PART` in the settings cell and run again (a new output dataset per part).